In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
df = pd.read_csv("../data/cleaned_data.csv")
print(df.shape)
df.head()

(50000, 27)


,age_years,gender,ssc_percentage,hsc_percentage,degree_percentage,degree_specialization,technical_score,aptitude_score,communication_score,skills_match_percentage,...,company_tier,job_role_match,competition_level,bond_requirement,notice_period_days,layoff_history,employment_gap_months,relocation_willingness,status,status_binary
0,27,Male,65.061656,83.842578,75.856526,Computer Science,58.221909,89.566305,64.474484,79.548913,...,Tier 3,Not Matched,Medium,Not Required,15.0,No,18.0,Not Willing,Not Placed,0
1,24,Male,67.885626,64.973305,73.093588,Electronics,71.927978,54.591971,61.077306,73.316134,...,Tier 1,Matched,High,Required,0.0,No,0.0,Not Willing,Not Placed,0
2,33,Female,73.892471,68.834121,90.196460,Information Technology,72.445041,58.587088,79.494739,75.466980,...,Tier 3,Not Matched,Low,Not Required,0.0,No,3.0,Not Willing,Placed,1
3,31,Male,74.145568,76.255126,75.586731,Mechanical,78.855676,61.022065,53.740386,73.676449,...,Tier 2,Matched,Low,Not Required,0.0,Yes,6.0,Not Willing,Not Placed,0
4,28,Male,60.475937,65.786336,80.801010,Information Technology,68.286776,65.713731,61.438314,88.994847,...,Tier 2,Matched,Medium,Not Required,0.0,No,3.0,Willing,Not Placed,0


In [4]:

df['interview_score'] = (
    df['technical_score'] + 
    df['aptitude_score'] + 
    df['communication_score']
) / 3


In [5]:
df['ctc_increase_pct'] = ((df['expected_ctc_lpa'] - df['previous_ctc_lpa'])
/ (df['previous_ctc_lpa'] + 1)) * 100


In [6]:
df['academic_avg'] = (df['ssc_percentage'] + df['hsc_percentage'] 
+ df['degree_percentage']) / 3

In [7]:
def experience_category(x):
    if x == 0:
        return "Fresher"
    elif x <= 3:
        return "Junior"
    else:
        return "Senior"
df['experience_level'] = df['years_of_experience'].apply(experience_category)

In [8]:
def skills_level(x):
    if x < 60:
        return "Low"
    elif x < 80:
        return "Medium"
    else:
        return "High"
df['skills_match_level'] = df['skills_match_percentage'].apply(skills_level)

In [9]:
df['salary_risk'] = (df['ctc_increase_pct'] > 75).astype(int)
df['gap_risk'] = (df['employment_gap_months'] > 12).astype(int)


In [10]:
numeric_features = ['age_years', 'ssc_percentage', 'hsc_percentage', 'degree_percentage', 
'technical_score', 'aptitude_score', 'communication_score', 
'skills_match_percentage', 'certifications_count', 'years_of_experience',
'previous_ctc_lpa', 'expected_ctc_lpa', 'notice_period_days',
'employment_gap_months', 'interview_score']


In [11]:
categorical_features = ['gender', 'degree_specialization', 'internship_experience', 
'career_switch_willingness', 'relevant_experience', 
'company_tier', 'job_role_match', 'competition_level',
'bond_requirement', 'layoff_history', 'relocation_willingness',
'experience_category', 'skills_match_level']

In [13]:
X_numeric = df[[col for col in numeric_features if col in df.columns]].fillna(0)
X_categorical = df[[col for col in categorical_features if col in df.columns]]
X_cat_encoded = pd.get_dummies(X_categorical, drop_first=True)

X_final = pd.concat([X_numeric, X_cat_encoded], axis=1)
print(f" Final X: {X_final.shape}")

 Final X: (50000, 33)


In [15]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_final), 
    columns=X_final.columns, 
    index=X_final.index
)
X_scaled.to_csv("../data/X_ml_ready.csv", index=False)
y.to_csv("../data/y_ml_ready.csv", index=False)